In [1]:
import os
import sys
import platform
from lakehouse import bronze, silver
from pyspark.sql import DataFrame, SparkSession
from delta import configure_spark_with_delta_pip
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from pyspark.testing import assertDataFrameEqual

In [2]:
if platform.system() == "Windows":
    os.environ["PYSPARK_PYTHON"] = sys.executable
    os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
    print("Adding Python ENV variables on Windows")

Adding Python ENV variables on Windows


In [3]:
builder = (
    SparkSession.builder.appName("Data with Nikk the Greek Spark Session")
    .master("local[4]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [4]:
CATALOG = spark.catalog.currentCatalog()

# 1. Set Up and Bronze Data

In [5]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")

DataFrame[]

In [6]:
options = {"catalog": CATALOG, "target_schema": "bronze"}

In [7]:
class TestBronze(bronze.Bronze):
    def custom_load(self, table):
        df = spark.range(10).withColumn("t", F.lit(table))
        return df


bronze_instance = TestBronze(spark, **options)

In [8]:
bronze_instance.load().transform().write(mode="overwrite").execute("people", "planets")

2025-02-27 16:32:17 | people | execute | Started
2025-02-27 16:32:17 | people | load | Started
2025-02-27 16:32:17 | people | load | Completed in 0.0 min
2025-02-27 16:32:17 | people | transform | Started
2025-02-27 16:32:17 | people | transform | Completed in 0.0 min
2025-02-27 16:32:17 | people | write | Started
2025-02-27 16:32:49 | people | write | Completed in 0.52 min
2025-02-27 16:32:49 | people | execute | Completed in 0.52 min
2025-02-27 16:32:49 | planets | execute | Started
2025-02-27 16:32:49 | planets | load | Started
2025-02-27 16:32:49 | planets | load | Completed in 0.0 min
2025-02-27 16:32:49 | planets | transform | Started
2025-02-27 16:32:49 | planets | transform | Completed in 0.0 min
2025-02-27 16:32:49 | planets | write | Started
2025-02-27 16:32:53 | planets | write | Completed in 0.07 min
2025-02-27 16:32:53 | planets | execute | Completed in 0.07 min


In [9]:
df = spark.sql(f"SELECT * FROM {CATALOG}.bronze.people")
print(f"No. Rows: {df.count()}")
df.show()

No. Rows: 10
+--------------------+---+------+
|         LH_BronzeTS| id|     t|
+--------------------+---+------+
|2025-02-27 16:32:...|  2|people|
|2025-02-27 16:32:...|  3|people|
|2025-02-27 16:32:...|  4|people|
|2025-02-27 16:32:...|  7|people|
|2025-02-27 16:32:...|  8|people|
|2025-02-27 16:32:...|  9|people|
|2025-02-27 16:32:...|  0|people|
|2025-02-27 16:32:...|  1|people|
|2025-02-27 16:32:...|  5|people|
|2025-02-27 16:32:...|  6|people|
+--------------------+---+------+



In [10]:
options = {
    "catalog": CATALOG,
    "source_schema": "bronze",
    "target_schema": "silver",
}

# 2 Debug with Overwrite example

In [11]:
class TestSilver(silver.Silver):
    def custom_filter(self, df: DataFrame, table: str) -> DataFrame:
        return df.where("id <= 5")

    def custom_transform(self, df: DataFrame, table: str) -> DataFrame:
        return df.withColumn("col", F.lit("col"))


silver_instance = TestSilver(spark, **options)

## 2.1 Debug the data load

In [12]:
# Debug without filter
silver_instance.load().execute("people")
actual_df = silver_instance.data["people"]
expected_df = (
    spark.range(10)
    .withColumn("t", F.lit("people"))
    .withColumn("LH_BronzeTS", F.current_timestamp())
)
actual_df.show(truncate=False)
# assertSchemaEqual(actual_df, expected_df)
assertDataFrameEqual(actual_df.drop("LH_BronzeTS"), expected_df.drop("LH_BronzeTS"))

2025-02-27 16:33:01 | people | execute | Started
2025-02-27 16:33:01 | people | load | Started
2025-02-27 16:33:01 | people | load | Completed in 0.0 min
2025-02-27 16:33:01 | people | execute | Completed in 0.0 min


+--------------------------+---+------+
|LH_BronzeTS               |id |t     |
+--------------------------+---+------+
|2025-02-27 16:32:19.916697|2  |people|
|2025-02-27 16:32:19.916697|3  |people|
|2025-02-27 16:32:19.916697|4  |people|
|2025-02-27 16:32:19.916697|7  |people|
|2025-02-27 16:32:19.916697|8  |people|
|2025-02-27 16:32:19.916697|9  |people|
|2025-02-27 16:32:19.916697|0  |people|
|2025-02-27 16:32:19.916697|1  |people|
|2025-02-27 16:32:19.916697|5  |people|
|2025-02-27 16:32:19.916697|6  |people|
+--------------------------+---+------+



In [13]:
# Debug with filter
silver_instance.load(filter="custom").execute("people")
actual_df = silver_instance.data["people"]
expected_df = (
    spark.range(10)
    .withColumn("t", F.lit("people"))
    .withColumn("LH_BronzeTS", F.current_timestamp())
    .where("id <= 5")
)
actual_df.show(truncate=False)
# assertSchemaEqual(actual_df, expected_df)
assertDataFrameEqual(actual_df.drop("LH_BronzeTS"), expected_df.drop("LH_BronzeTS"))

2025-02-27 16:33:03 | people | execute | Started
2025-02-27 16:33:03 | people | load | Started
2025-02-27 16:33:03 | people | load | Completed in 0.0 min
2025-02-27 16:33:03 | people | execute | Completed in 0.0 min


+--------------------------+---+------+
|LH_BronzeTS               |id |t     |
+--------------------------+---+------+
|2025-02-27 16:32:19.916697|2  |people|
|2025-02-27 16:32:19.916697|3  |people|
|2025-02-27 16:32:19.916697|4  |people|
|2025-02-27 16:32:19.916697|0  |people|
|2025-02-27 16:32:19.916697|1  |people|
|2025-02-27 16:32:19.916697|5  |people|
+--------------------------+---+------+



# 2.2 Debug transformation

In [14]:
# Debug with default transformation
silver_instance.load(filter="custom").transform().execute("people")
actual_df = silver_instance.data["people"]
expected_df = (
    spark.range(10)
    .withColumn("t", F.lit("people"))
    .withColumn("LH_BronzeTS", F.current_timestamp())
    .where("id <= 5")
    .withColumn("col", F.lit("col"))
    .withColumn("LH_SilverTS", F.current_timestamp())
)
actual_df.show(truncate=False)
# assertSchemaEqual(actual_df, expected_df)
assertDataFrameEqual(
    actual_df.drop("LH_BronzeTS", "LH_SilverTS"),
    expected_df.drop("LH_BronzeTS", "LH_SilverTS"),
)

2025-02-27 16:33:07 | people | execute | Started
2025-02-27 16:33:07 | people | load | Started
2025-02-27 16:33:07 | people | load | Completed in 0.0 min
2025-02-27 16:33:07 | people | transform | Started
2025-02-27 16:33:07 | people | transform | Completed in 0.0 min
2025-02-27 16:33:07 | people | execute | Completed in 0.0 min


+--------------------------+--------------------------+---+------+---+
|LH_SilverTS               |LH_BronzeTS               |id |t     |col|
+--------------------------+--------------------------+---+------+---+
|2025-02-27 16:33:07.383396|2025-02-27 16:32:19.916697|2  |people|col|
|2025-02-27 16:33:07.383396|2025-02-27 16:32:19.916697|3  |people|col|
|2025-02-27 16:33:07.383396|2025-02-27 16:32:19.916697|4  |people|col|
|2025-02-27 16:33:07.383396|2025-02-27 16:32:19.916697|0  |people|col|
|2025-02-27 16:33:07.383396|2025-02-27 16:32:19.916697|1  |people|col|
|2025-02-27 16:33:07.383396|2025-02-27 16:32:19.916697|5  |people|col|
+--------------------------+--------------------------+---+------+---+



In [15]:
# Debug without default transformation
silver_instance.load(filter="custom").transform(ignore_defaults=True).execute("people")
actual_df = silver_instance.data["people"]
expected_df = (
    spark.range(10)
    .withColumn("t", F.lit("people"))
    .withColumn("LH_BronzeTS", F.current_timestamp())
    .where("id <= 5")
    .withColumn("col", F.lit("col"))
)
actual_df.show(truncate=False)
# assertSchemaEqual(actual_df, expected_df)
assertDataFrameEqual(actual_df.drop("LH_BronzeTS"), expected_df.drop("LH_BronzeTS"))

2025-02-27 16:33:08 | people | execute | Started
2025-02-27 16:33:08 | people | load | Started
2025-02-27 16:33:08 | people | load | Completed in 0.0 min
2025-02-27 16:33:08 | people | transform | Started
2025-02-27 16:33:08 | people | transform | Completed in 0.0 min
2025-02-27 16:33:08 | people | execute | Completed in 0.0 min


+--------------------------+---+------+---+
|LH_BronzeTS               |id |t     |col|
+--------------------------+---+------+---+
|2025-02-27 16:32:19.916697|2  |people|col|
|2025-02-27 16:32:19.916697|3  |people|col|
|2025-02-27 16:32:19.916697|4  |people|col|
|2025-02-27 16:32:19.916697|0  |people|col|
|2025-02-27 16:32:19.916697|1  |people|col|
|2025-02-27 16:32:19.916697|5  |people|col|
+--------------------------+---+------+---+



# 2.3 Debug write

In [16]:
silver_instance.load(filter="custom").transform().write(mode="overwrite").execute(
    "people"
)
actual_df = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
expected_df = (
    spark.range(10)
    .withColumn("t", F.lit("people"))
    .withColumn("LH_BronzeTS", F.current_timestamp())
    .where("id <= 5")
    .withColumn("col", F.lit("col"))
    .withColumn("LH_SilverTS", F.current_timestamp())
)
actual_df.show(truncate=False)
# assertSchemaEqual(actual_df, expected_df)
assertDataFrameEqual(
    actual_df.drop("LH_BronzeTS", "LH_SilverTS"),
    expected_df.drop("LH_BronzeTS", "LH_SilverTS"),
)

2025-02-27 16:33:10 | people | execute | Started
2025-02-27 16:33:10 | people | load | Started
2025-02-27 16:33:10 | people | load | Completed in 0.0 min
2025-02-27 16:33:10 | people | transform | Started
2025-02-27 16:33:10 | people | transform | Completed in 0.0 min
2025-02-27 16:33:10 | people | write | Started
2025-02-27 16:33:14 | people | write | Completed in 0.07 min
2025-02-27 16:33:14 | people | execute | Completed in 0.07 min


+--------------------------+--------------------------+---+------+---+
|LH_SilverTS               |LH_BronzeTS               |id |t     |col|
+--------------------------+--------------------------+---+------+---+
|2025-02-27 16:33:10.569962|2025-02-27 16:32:19.916697|2  |people|col|
|2025-02-27 16:33:10.569962|2025-02-27 16:32:19.916697|3  |people|col|
|2025-02-27 16:33:10.569962|2025-02-27 16:32:19.916697|4  |people|col|
|2025-02-27 16:33:10.569962|2025-02-27 16:32:19.916697|0  |people|col|
|2025-02-27 16:33:10.569962|2025-02-27 16:32:19.916697|1  |people|col|
|2025-02-27 16:33:10.569962|2025-02-27 16:32:19.916697|5  |people|col|
+--------------------------+--------------------------+---+------+---+



# 3 Clean Up

In [17]:
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.bronze CASCADE")
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.silver CASCADE")

DataFrame[]